In [4]:
%pip install zxcvbn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from importlib.metadata import version

print("zxcvbn version:", version("zxcvbn"))

zxcvbn version: 4.5.0


In [5]:
from zxcvbn import zxcvbn

# Test the model with an example password
result = zxcvbn("password123")

print("Estimated guesses:", result["guesses"])
print("Log10 guesses:", result["guesses_log10"])
print("Strength score:", result["score"])

Estimated guesses: 596
Log10 guesses: 2.775246259740236
Strength score: 0


In [7]:
import pandas as pd

# Load the balanced test dataset
test_df = pd.read_csv(
    "../data/processed/balanced_test.csv",
    dtype={"password": "string"},
    keep_default_na=False
)

# Check the dataset
print("Total passwords:", len(test_df))
print("\nPasswords per group:")
print(test_df["group"].value_counts())

print("\nFirst five rows:")
display(test_df.head())

Total passwords: 73220

Passwords per group:
group
long     36610
short    36610
Name: count, dtype: int64

First five rows:


,rank,password,length,group
0,812637,Z087uDYZPF,10,long
1,641090,amnez1983,9,long
2,874568,xrfkjdf24,9,long
3,383793,w1retl,6,short
4,791110,Zenit1925,9,long


In [8]:
# Test zxcvbn on the first 10 passwords
sample_df = test_df.head(10).copy()

# Estimate password strength
sample_df["guesses"] = sample_df["password"].apply(
    lambda password: zxcvbn(password)["guesses"]
)

sample_df["score"] = sample_df["password"].apply(
    lambda password: zxcvbn(password)["score"]
)

# Display results
display(sample_df[["password", "length", "group", "guesses", "score"]])

,password,length,group,guesses,score
0,Z087uDYZPF,10,long,10000000001,3
1,amnez1983,9,long,10010000,2
2,xrfkjdf24,9,long,1000000001,3
3,w1retl,6,short,675200,1
4,Zenit1925,9,long,5088032,2
5,JESUCRIST,9,long,22569,1
6,crusaders,9,long,6598,1
7,cherokee1,9,long,21264,1
8,zee8zw,6,short,1000001,1
9,halo1234,8,short,356654,1


In [10]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from tqdm.auto import tqdm

# Enable progress bar for pandas
tqdm.pandas()

# Run zxcvbn once for each password
results = test_df["password"].progress_apply(zxcvbn)

# Extract relevant estimates
test_df["zxcvbn_guesses"] = results.apply(lambda r: r["guesses"])
test_df["zxcvbn_log10_guesses"] = results.apply(
    lambda r: r["guesses_log10"]
)
test_df["zxcvbn_score"] = results.apply(lambda r: r["score"])

print("Passwords evaluated:", len(test_df))
print("\nScore distribution:")
print(test_df["zxcvbn_score"].value_counts().sort_index())

c:\Users\jessi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 73220/73220 [00:29<00:00, 2467.66it/s]


Passwords evaluated: 73220

Score distribution:
zxcvbn_score
0      963
1    35270
2    16810
3    13233
4     6944
Name: count, dtype: int64


In [12]:
def evaluate_zxcvbn(passwords):
    results = []

    for password in passwords:
        result = zxcvbn(password)

        results.append({
            "guesses": result["guesses"],
            "log10_guesses": result["guesses_log10"],
            "score": result["score"]
        })

    return pd.DataFrame(results)

In [13]:
test_results = evaluate_zxcvbn(["password123", "JESUCRIST", "w1retl"])

display(test_results)

,guesses,log10_guesses,score
0,596,2.775246,0
1,22569,4.353512,1
2,675200,5.829432,1


In [14]:
print("Total passwords:", len(test_df))

print("\nMissing results:")
print(
    test_df[
        ["zxcvbn_guesses", "zxcvbn_log10_guesses", "zxcvbn_score"]
    ].isna().sum()
)

Total passwords: 73220

Missing results:
zxcvbn_guesses          0
zxcvbn_log10_guesses    0
zxcvbn_score            0
dtype: int64


In [15]:
import sys
sys.path.insert(0, "../src")

from zxcvbn_model import evaluate_zxcvbn

display(evaluate_zxcvbn(["password123", "JESUCRIST", "w1retl"]))

,guesses,log10_guesses,score
0,596,2.775246,0
1,22569,4.353512,1
2,675200,5.829432,1
